# ARC LB33 + TRM parity v1 — failure-safe repair

This copy preserves the 4B sampling, TTT, DFS, selector, and gap-fill merge policy. It repairs phase budgeting, worker shutdown, candidate-file integrity, TRM test-index alignment, and final submission durability. The latest teammate run was cancelled; its log contained no traceback.


In [1]:
# Reserve a real Phase-B window while keeping Kaggle's final 10-minute write buffer.
import os
import time

kernel_start_time = time.time()
global_end_time = kernel_start_time + 12 * 3600 - 600
rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
phase_b_reserve_s = 4200.0 if rerun_mode else 1800.0
baseline_end_time = global_end_time - phase_b_reserve_s
print(
    f"[budget] baseline={baseline_end_time-kernel_start_time:.0f}s "
    f"phase_b_reserve={phase_b_reserve_s:.0f}s final_write_buffer=600s",
    flush=True,
)


[budget] baseline=40800s phase_b_reserve=1800s final_write_buffer=600s


In [2]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [3]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [4]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            return
        for key in sorted(os.listdir(store)):
            path = os.path.join(store, key)
            if not os.path.isfile(path) or key.startswith(".") or ".tmp." in key:
                continue
            try:
                with bz2.BZ2File(path) as f:
                    outputs = pickle.load(f)
                if not isinstance(outputs, list):
                    raise TypeError(f"decoded payload is {type(outputs).__name__}, expected list")
            except (OSError, EOFError, pickle.UnpicklingError, TypeError, ValueError) as exc:
                print(f"*** Skip corrupt candidate file {key!r}: {exc!r}")
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                if not isinstance(sample, dict) or "solution" not in sample:
                    print(f"*** Skip malformed candidate {key!r} #{i}")
                    continue
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(" avg/max correct beam score: n/a (no correct candidate)")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

Writing arc_decoder.py


In [5]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!", flush=True)
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        target = os.path.join(dir_outputs, subkey)
                        temp = os.path.join(dir_outputs, f".{subkey}.tmp.{os.getpid()}")
                        try:
                            with bz2.BZ2File(temp, "w") as f:
                                pickle.dump(decoded_result, f)
                            os.replace(temp, target)
                        finally:
                            if os.path.exists(temp):
                                os.remove(temp)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

Writing arc_solver.py


In [6]:
%%writefile starter.py
import glob
import json
import os
import time
import torch
import argparse
import torch.multiprocessing as mp


MARKER_ROOT = "/kaggle/arc_lb33_worker_markers"


def _marker(rank, suffix):
    return os.path.join(MARKER_ROOT, f"worker{rank}.{suffix}")


def _write_marker(rank, suffix, text):
    os.makedirs(MARKER_ROOT, exist_ok=True)
    target = _marker(rank, suffix)
    temp = target + f".tmp.{os.getpid()}"
    with open(temp, "w", encoding="utf-8") as handle:
        handle.write(text)
    os.replace(temp, target)


def local_worker(rank, queue, end_time):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["PYTHONUNBUFFERED"] = "1"
    torch.set_default_device("cpu")

    try:
        if rank > 0:
            wait_deadline = min(end_time, time.time() + float(os.getenv("ARC_INIT_CHAIN_S", "1200")))
            while not os.path.exists(_marker(rank - 1, "ready")):
                if os.path.exists(_marker(rank - 1, "failed")):
                    raise RuntimeError(f"previous worker {rank-1} failed during initialization")
                if time.time() >= wait_deadline:
                    raise TimeoutError(f"timed out waiting for worker {rank-1} initialization")
                time.sleep(2)

        print(f"[Rank {rank}] importing solver", flush=True)
        from arc_solver import worker
        _write_marker(rank, "ready", "OK")
        print(f"[Rank {rank}] start", flush=True)
        worker(rank, queue, end_time)
        print(f"[Rank {rank}] done", flush=True)
    except BaseException as exc:
        _write_marker(rank, "failed", repr(exc))
        print(f"[Rank {rank}] failed: {exc!r}", flush=True)
        raise


def _terminate_and_join(processes):
    alive = [process for process in processes if process.is_alive()]
    for process in alive:
        process.terminate()
    for process in alive:
        process.join(timeout=15)
    still_alive = [process for process in alive if process.is_alive()]
    for process in still_alive:
        process.kill()
    for process in still_alive:
        process.join(timeout=5)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, required=True)
    args = parser.parse_args()

    rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r", encoding="utf-8") as handle:
        data = json.load(handle)

    gpu_count = torch.cuda.device_count()
    if gpu_count < 1:
        raise RuntimeError("GPU preflight failed: no CUDA device is visible")
    nprocs = min(4, gpu_count)
    print(f"[starter] gpu_count={gpu_count} workers={nprocs} rerun={rerun_mode}", flush=True)

    os.makedirs(MARKER_ROOT, exist_ok=True)
    for old in glob.glob(os.path.join(MARKER_ROOT, "worker*")):
        try:
            os.remove(old)
        except FileNotFoundError:
            pass

    manager = mp.Manager()
    queue = manager.Queue()
    for key in sorted(data):
        if not rerun_mode and key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
            continue
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    context = mp.spawn(local_worker, args=(queue, args.end_time), nprocs=nprocs, join=False)
    outdir = "/kaggle/inference_outputs"
    stall_cap = float(os.getenv("ARC_STALL_S", "2400"))
    hard_end = args.end_time
    last_count = -1
    last_progress = time.time()
    reason = "complete"
    try:
        while True:
            try:
                if context.join(timeout=30):
                    break
            except Exception as exc:
                reason = f"worker failure: {exc!r}"
                print(f"[starter] {reason}; keeping partial results", flush=True)
                break

            count = len([
                path for path in glob.glob(os.path.join(outdir, "*"))
                if os.path.isfile(path) and ".tmp." not in os.path.basename(path)
            ])
            if count != last_count:
                last_count = count
                last_progress = time.time()
                print(f"[starter] progress files={count}", flush=True)

            now = time.time()
            if now >= hard_end:
                reason = "phase deadline"
                break
            if now - last_progress >= stall_cap:
                reason = f"no output progress for {stall_cap:.0f}s"
                break
    finally:
        _terminate_and_join(context.processes)
        manager.shutdown()
    print(f"[starter] exit reason={reason} files={max(last_count, 0)}", flush=True)


Writing starter.py


In [7]:
!PYTHONUNBUFFERED=1 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python -u starter.py --end-time {baseline_end_time}

[starter] gpu_count=4 workers=4 rerun=False
[Rank 0] importing solver
[starter] progress files=0
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 0] start
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading checkpoint shards:   0%|                          | 0/2 [00:00<?, ?it/s][Rank 1] importing solver
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Loading checkpoint shards: 100%|██████████████████| 2/2 [01:14<00:00, 37.16s/it]
[Rank 1] start
==((====))==  Unslot

In [8]:
# ---------- Phase B (additive): bounded TRM test-time training ----------
# Any Phase-B failure leaves the already-produced 4B candidates untouched.
import glob
import os
import shutil
import signal
import subprocess
import sys
import time

rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))


def _first_valid_root(candidates, required):
    seen = set()
    for root in candidates:
        root = os.path.realpath(root)
        if root in seen:
            continue
        seen.add(root)
        if all(os.path.isfile(os.path.join(root, item)) for item in required):
            return root
    return None


bundle_candidates = [
    "/kaggle/input/trm-bundle",
    "/kaggle/input/datasets/koushikrudra/trm-bundle",
]
bundle_candidates += [
    os.path.dirname(path)
    for pattern in ("/kaggle/input/*/trm_driver.py", "/kaggle/input/*/*/trm_driver.py")
    for path in sorted(glob.glob(pattern))
]
TRM_BUNDLE = _first_valid_root(
    bundle_candidates,
    ("trm_driver.py", "trm_build_data.py", "trm_eval_lib.py", "evaluators/arc.py"),
)

checkpoint_candidates = [
    "/kaggle/input/arc-prize-trm-031",
    "/kaggle/input/datasets/cpmpml/arc-prize-trm-031",
]
checkpoint_candidates += sorted(glob.glob("/kaggle/input/*arc-prize-trm-031*"))
TRM_CKPT_DIR = None
TRM_CKPT_STEP = None
for root in checkpoint_candidates:
    steps = []
    for path in glob.glob(os.path.join(root, "**", "step_*"), recursive=True):
        if not os.path.isfile(path):
            continue
        try:
            step = int(os.path.basename(path).split("_", 1)[1])
        except (IndexError, ValueError):
            continue
        steps.append((step, path))
    if steps:
        _, latest = max(steps)
        TRM_CKPT_DIR = os.path.dirname(latest)
        TRM_CKPT_STEP = os.path.basename(latest)
        break

HAS_TRM = TRM_BUNDLE is not None and TRM_CKPT_DIR is not None
TRM_BUDGET = float(os.getenv("TRM_BUDGET_S", "1800"))
TRM_NAUG = os.getenv("TRM_NAUG", "128")
if not rerun_mode:
    TRM_BUDGET = min(TRM_BUDGET, 900.0)
    TRM_NAUG = os.getenv("TRM_NAUG", "16")

print(
    f"[trm-orch] HAS_TRM={HAS_TRM} bundle={TRM_BUNDLE} "
    f"ckpt_dir={TRM_CKPT_DIR} ckpt_step={TRM_CKPT_STEP}",
    flush=True,
)


def _run_bounded(command, *, cwd, env, timeout):
    print(f"[trm-orch] exec timeout={timeout:.0f}s: {' '.join(command)}", flush=True)
    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=True)
    try:
        returncode = process.wait(timeout=max(1.0, timeout))
    except subprocess.TimeoutExpired:
        print(f"[trm-orch] timeout; terminating process group {process.pid}", flush=True)
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=10)
        raise
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


minimum_window = 3000.0
# The checkpoint training metadata contains all 120 public-evaluation tasks.
# A normal Kaggle commit therefore must not run/score TRM on that contaminated
# split. Keep Phase B armed only for an explicitly approved competition rerun.
if rerun_mode and HAS_TRM and global_end_time - time.time() > minimum_window:
    try:
        work = "/kaggle/working/trm_code"
        if os.path.isdir(work):
            shutil.rmtree(work)
        shutil.copytree(TRM_BUNDLE, work)

        # Remove incompatible vendored binary wheels; use the notebook environment's packages.
        python_tag = f"cpython-{sys.version_info[0]}{sys.version_info[1]}"
        vendored_sos = glob.glob(os.path.join(work, "pydantic_core", "*.so"))
        if vendored_sos and not any(python_tag in path for path in vendored_sos):
            for directory in ("pydantic_core", "pydantic", "annotated_types", "typing_inspection"):
                shutil.rmtree(os.path.join(work, directory), ignore_errors=True)
            print(f"[trm-orch] stripped vendored pydantic (ABI != {python_tag})", flush=True)

        # Normalize torch.compile checkpoint key prefixes. Fail closed if the bundle changed.
        loader_path = os.path.join(work, "trm_eval_lib.py")
        loader_source = open(loader_path, encoding="utf-8").read()
        loader_anchor = (
            '        # Resize and reset puzzle emb if needed\n'
            '        puzzle_emb_name = "_orig_mod.model.inner.puzzle_emb.weights"\n'
        )
        loader_patch = (
            '        _compiled = hasattr(model, "_orig_mod")\n'
            '        def _norm(k):\n'
            '            k2 = k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k\n'
            '            return ("_orig_mod." + k2) if _compiled else k2\n'
            '        state_dict = {_norm(k): v for k, v in state_dict.items()}\n'
            '        # Resize and reset puzzle emb if needed\n'
            '        puzzle_emb_name = _norm("_orig_mod.model.inner.puzzle_emb.weights")\n'
        )
        if loader_source.count(loader_anchor) != 1:
            raise RuntimeError("TRM checkpoint-loader anchor mismatch")
        with open(loader_path, "w", encoding="utf-8") as handle:
            handle.write(loader_source.replace(loader_anchor, loader_patch, 1))

        # Preserve test-pair indices when a TRM prediction is missing.
        evaluator_path = os.path.join(work, "evaluators", "arc.py")
        evaluator_source = open(evaluator_path, encoding="utf-8").read()
        evaluator_anchor = (
            '                if not len(p_map):\n'
            '                    print (f"Puzzle {name} has no predictions.")\n'
            '                    continue\n'
        )
        evaluator_patch = (
            '                if not len(p_map):\n'
            '                    print (f"Puzzle {name} has no predictions.")\n'
            '                    submission[name].append({})\n'
            '                    continue\n'
        )
        if evaluator_source.count(evaluator_anchor) != 1:
            raise RuntimeError("TRM evaluator index-preservation anchor mismatch")
        with open(evaluator_path, "w", encoding="utf-8") as handle:
            handle.write(evaluator_source.replace(evaluator_anchor, evaluator_patch, 1))

        if rerun_mode:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        else:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        remaining = global_end_time - time.time()
        build_cap = 2100.0 if rerun_mode else 1200.0
        build_timeout = min(build_cap, remaining - TRM_BUDGET - 300.0)
        if build_timeout < 300.0:
            raise TimeoutError(f"insufficient dataset-build window: {build_timeout:.0f}s")
        _run_bounded(
            [sys.executable, "-u", "trm_build_data.py", test_path,
             "/kaggle/working/trm_data", str(TRM_NAUG)],
            cwd=work,
            env=env,
            timeout=build_timeout,
        )

        import torch as _trm_torch
        ngpu = max(1, _trm_torch.cuda.device_count())
        if rerun_mode and ngpu != 4:
            raise RuntimeError(f"TRM requires the expected L4x4 allocation; visible GPUs={ngpu}")

        remaining = global_end_time - time.time()
        driver_budget = min(TRM_BUDGET, remaining - 300.0)
        if driver_budget < 600.0:
            raise TimeoutError(f"insufficient TRM driver window: {driver_budget:.0f}s")
        eval_reserve = min(1500.0 if rerun_mode else 600.0, max(300.0, driver_budget - 300.0))
        trm_env = env.copy()
        trm_env.update({
            "TRM_DATA_DIR": "/kaggle/working/trm_data",
            "TRM_CKPT_DIR": TRM_CKPT_DIR,
            "TRM_CKPT_STEP": TRM_CKPT_STEP,
            "TRM_OUT_DIR": "/kaggle/trm_out",
            "TRM_BUDGET_S": str(driver_budget),
            "TRM_EVAL_RESERVE_S": str(eval_reserve),
        })
        if not rerun_mode:
            ngpu = 1
            trm_env["DISABLE_COMPILE"] = "1"
            trm_env["TRM_BATCH"] = env.get("TRM_BATCH", "32")

        if ngpu == 1:
            command = [sys.executable, "-u", "trm_driver.py"]
        else:
            command = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone",
                       f"--nproc_per_node={ngpu}", "trm_driver.py"]
        driver_timeout = min(global_end_time - time.time() - 180.0, driver_budget + 300.0)
        _run_bounded(command, cwd=work, env=trm_env, timeout=driver_timeout)
        print("[trm-orch] Phase B complete", flush=True)
    except Exception as exc:
        print(f"[trm-orch] Phase B failed; baseline remains usable: {exc!r}", flush=True)
else:
    reason = (
        "public-eval contamination guard (competition rerun only)"
        if not rerun_mode
        else "inputs missing or insufficient reserved time"
    )
    print(f"[trm-orch] skipped: {reason}", flush=True)


[trm-orch] HAS_TRM=True bundle=/kaggle/input/datasets/koushikrudra/trm-bundle ckpt_dir=/kaggle/input/datasets/cpmpml/arc-prize-trm-031 ckpt_step=step_275886
[trm-orch] skipped: public-eval contamination guard (competition rerun only)


In [9]:
import os
import json
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

os.makedirs("/kaggle/inference_outputs", exist_ok=True)  # survive an engine crash
try:
    decoder.load_decoded_results("/kaggle/inference_outputs")
    selected = decoder.run_selection_algo()
except Exception as exc:
    print(f"*** Baseline decode/selection failed; using submission skeleton: {exc!r}")
    selected = {}

# ---- TRM union: strictly gap-filling second source (never displaces 4B) ----
trm_preds = {}
try:
    if os.path.isfile("/kaggle/trm_out/trm_predictions.json"):
        with open("/kaggle/trm_out/trm_predictions.json") as f:
            trm_preds = json.load(f)
        print(f"*** TRM predictions loaded for {len(trm_preds)} tasks")
except Exception as e:
    print(f"*** TRM predictions unavailable: {e!r}")

def _clean_trm(preds):
    clean = {}
    if not isinstance(preds, dict):
        return clean
    for _name, _pairs in preds.items():
        if not isinstance(_pairs, list) or _name not in data.queries:
            continue
        ok_pairs = []
        for _att in _pairs:
            entry = {}
            if isinstance(_att, dict):
                for _k, _v in _att.items():
                    if not (isinstance(_k, str) and _k.startswith("attempt_")
                            and _k.split("_", 1)[1].isdigit()):
                        continue
                    try:
                        _a = np.asarray(_v, dtype=np.int64)
                        if (_a.ndim == 2 and 1 <= _a.shape[0] <= 30
                                and 1 <= _a.shape[1] <= 30
                                and _a.min() >= 0 and _a.max() <= 9):
                            entry[_k] = _a.tolist()
                    except Exception:
                        pass
            ok_pairs.append(entry)
        clean[_name] = ok_pairs[:len(data.queries[_name]["test"])]
    return clean

trm_preds = _clean_trm(trm_preds)

def _distinct(grids):
    seen, out = set(), []
    for g in grids:
        arr = np.asarray(g, dtype=np.uint8)
        h = (arr.shape, arr.tobytes())
        if h not in seen:
            seen.add(h)
            out.append(arr)
    return out

merged = dict(selected)
n_fill2 = n_fill_both = 0
for name, pairs in trm_preds.items():
    for i, attempts in enumerate(pairs):
        bk = f"{name}_{i}"
        t_grids = _distinct([attempts[k] for k in sorted(attempts,
                            key=lambda x: int(x.split("_")[1])) if k.startswith("attempt_")])
        if not t_grids:
            continue
        ranked = _distinct(merged.get(bk, []))
        if len(ranked) >= 2:
            continue
        if len(ranked) == 1:
            extra = next((t for t in t_grids
                          if not (t.shape == ranked[0].shape and np.array_equal(t, ranked[0]))), None)
            if extra is not None:
                merged[bk] = [ranked[0], extra]
                n_fill2 += 1
        else:
            merged[bk] = t_grids[:2]
            n_fill_both += 1
print(f"*** union merge: attempt2-filled={n_fill2}, both-filled={n_fill_both}")

submission = data.get_submission(merged)

# ---- final guard: only valid grids reach submission.json ----
def _valid_grid(g):
    if not isinstance(g, list) or not (1 <= len(g) <= 30):
        return False
    if not all(isinstance(r, list) and 1 <= len(r) <= 30 for r in g):
        return False
    width = len(g[0])
    return (all(len(r) == width for r in g) and
            all(isinstance(x, int) and not isinstance(x, bool) and 0 <= x <= 9
                for r in g for x in r))

_n_bad = 0
for _k, _entries in submission.items():
    for _e in _entries:
        for _a in ("attempt_1", "attempt_2"):
            if not _valid_grid(_e.get(_a)):
                _e[_a] = [[0]]
                _n_bad += 1
print(f"*** submission: {len(submission)} tasks, repaired_grids={_n_bad}")

submission_path = "/kaggle/working/submission.json"
submission_temp = submission_path + ".tmp"
with open(submission_temp, "w", encoding="utf-8") as f:
    json.dump(submission, f)
os.replace(submission_temp, submission_path)
print(f"*** submission atomically written: {submission_path}")

if not rerun_mode:
    try:
        decoder.benchmark_selection_algos()
        with open(submission_path, "r", encoding="utf-8") as f:
            reload_submission = json.load(f)
        print("*** Reload score:", data.validate_submission(reload_submission))
    except Exception as exc:
        print(f"*** Optional smoke benchmark failed after safe submission write: {exc!r}")

*** Load solutions from '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json'...
*** union merge: attempt2-filled=0, both-filled=0
*** Generating submission for 5 outputs...
*** submission: 120 tasks, repaired_grids=0
*** submission atomically written: /kaggle/working/submission.json
*** Benchmark selection algorithms...
ALL_CORRECT: 0.00001 -  3.28604 16x16 [36a08778_0.permute4150723698.ex031524.out0]
ALL_CORRECT: 0.00001 -  3.28604 16x16 [36a08778_0.permute6720984531.ex021543.out0]
ALL_CORRECT: 0.05983 -  3.28604 16x16 [36a08778_0.rot90.permute1258796043.ex532401.out0]
ALL_CORRECT: 0.01260 -  3.28604 16x16 [36a08778_0.rot90.permute5386701942.ex542013.out0]
ALL_CORRECT: 0.14230 -  3.28604 16x16 [36a08778_0.rot90.rot90.permute7931086254.ex405132.out0]
ALL_CORRECT: 0.20652 -  3.28604 16x16 [36a08778_0.rot90.rot90.permute9302675184.ex105324.out0]
ALL_CORRECT: 0.00410 -  3.28604 16x16 [36a08778_0.transpose.permute4950812673.ex132450.out0]
ALL_CORRECT: 0.0